# M17 L17: Product Metrics + SQL/Python for AI Products

**Build an 8-metric AI-product analytics stack: events, metrics, A/B test, funnel.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

An AI product is measured on three axes: **usage**, **quality**, and **cost**.

- **Usage**: do people use it? (DAU = daily active users, adoption = % who tried it, engagement = how often, retention = did they come back?)
- **Quality**: does it work? (RAGAS from M10, and "did it say I don't know?")
- **Cost**: is it cheap? (cents per user, milliseconds per response)

For every metric, you need TWO definitions: one in SQL (so the warehouse can compute it on millions of events) and one in Python (so the eval pipeline can compute it on a small sample). If they disagree, you have a bug.

The lesson also builds an **A/B test** (control vs treatment, lift, p-value) and a **funnel** (where do users drop off?).

## The 5 things to know

1. **Three axes**: usage (DAU/WAU/MAU, adoption, engagement, retention), quality (RAGAS, task success), cost (cents/user, latency P95).
2. **SQL + Python duality**: every metric needs both definitions; they must agree.
3. **Per-user rates, not totals**: a total grows with users and tells you nothing; per-user rates are comparable week-over-week.
4. **A/B test wiring**: deterministic variant by hash; lift = (treatment - control) / control; p-value tells you if the lift is noise.
5. **Funnel reveals the biggest opportunity**: the largest drop-off is where to focus engineering effort.

In [ ]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm17-product-metrics' / 'L17-product-metrics.py'
_spec = importlib.util.spec_from_file_location('L17', _p)
_L17 = importlib.util.module_from_spec(_spec)
sys.modules['L17'] = _L17
_spec.loader.exec_module(_L17)

print(f'Setup OK. Lesson module: {_L17.LESSON_NUMBER}')

In [ ]:
# Run the 17 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L17, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

In [ ]:
# === Section 1: synthetic event store ===
from datetime import datetime

store = _L17._build_synthetic_store()
today = datetime(2026, 10, 15)
print(f'Total events: {len(store)}')
print(f'Unique users: {len(store.unique_users())}')
by_name = {}
for e in store.events:
    by_name[e.event_name] = by_name.get(e.event_name, 0) + 1
for name, n in sorted(by_name.items()):
    print(f'  {name:<14} {n}')

In [ ]:
# === Section 2: metric definitions ===
print(f'\n{"metric":<18}  {"unit":<8}')
print('-' * 28)
for m in _L17.METRICS:
    print(f'  {m.name:<18}  {m.unit:<8}')

In [ ]:
# === Section 3: compute all 10 metrics ===
print(f'\n{"metric":<18}  {"value":>10}   {"unit":<8}')
print('-' * 40)
for m in _L17.METRICS:
    v = _L17.compute_metric(m, store, today)
    print(f'  {m.name:<18}  {v:>10.3f}   {m.unit:<8}')

In [ ]:
# === Section 4: SQL sample for adoption_rate ===
m = next(x for x in _L17.METRICS if x.name == 'adoption_rate')
print(f'\nSQL definition for {m.name}:\n')
for line in m.sql_definition.split('\n'):
    print(f'  {line}')

In [ ]:
# === Section 5: A/B test ===
ab = _L17.ABTest(name='ai_quality_v2', salt='2026-Q4')
report = ab.run(store, metric='ragas_score')
print(f'\nA/B test report:')
print(f'  name:           {report["name"]}')
print(f'  metric:         {report["metric"]}')
print(f'  control mean:   {report["control_mean"]:.3f}')
print(f'  treatment mean: {report["treatment_mean"]:.3f}')
print(f'  lift:           {report["lift"]:+.1%}')
print(f'  p-value:        {report["p_value"]:.3f}')
print(f'  significant:    {report["significant"]}')

In [ ]:
# === Section 6: funnel ===
funnel = _L17.Funnel(
    name='ai_value_funnel',
    steps=[['login'], ['view_ai'], ['ai_query'], ['ai_thumbs_up']],
)
a = funnel.analyze(store)
step_names = ['login', 'view_ai', 'ai_query', 'thumbs_up']
print(f'\nFunnel: {funnel.name}')
print(f'  {"step":<14}  {"users":>6}  {"drop-off":>10}')
for i, n in enumerate(a['steps']):
    drop = a['drop_offs'][i-1] if i > 0 else 0.0
    print(f'  {step_names[i]:<14}  {n:>6}  {drop:>10.1%}')
print(f'\n  overall conversion: {a["conversion"]:.1%}')

In [ ]:
# === Section 7: latency percentiles ===
queries = store.query('ai_query')
latencies = [float(e.properties['latency_ms']) for e in queries if 'latency_ms' in e.properties]
print(f'\nLatency percentiles (n={len(latencies)}):')
print(f'  p50:  {_L17._percentile(latencies, 0.50):>7.1f} ms')
print(f'  p95:  {_L17._percentile(latencies, 0.95):>7.1f} ms')
print(f'  p99:  {_L17._percentile(latencies, 0.99):>7.1f} ms')

## Architecture (Mermaid)

```mermaid
flowchart LR
    E[Events:<br/>user_id, event_name, timestamp, properties] --> ES[EventStore]
    ES --> M[8 Metrics:<br/>SQL def + Python impl]
    M --> R[Report]
    R --> AB[A/B test:<br/>lift + p-value]
    R --> F[Funnel:<br/>drop-off per step]
    R --> D[Dashboard]
    E --> ET[ETL to warehouse]
    ET --> SQL[SQL metrics<br/>on Snowflake]
    SQL --> D
```

**Read as:** events flow into the EventStore; metrics are computed in Python (fast iteration on samples) and SQL (warehouse on full data); the report feeds the A/B test, the funnel, and the dashboard.

## Reflect (10 min)

1. Why do we need BOTH a SQL and a Python definition for each metric? What happens if they disagree?
2. The lesson reports `engagement` as queries / active_user / day. Why per-user rates instead of totals?
3. The mock p-value is `exp(-diff * 10)`. What does this imply about how confident you should be in the result?
4. The funnel shows a 41% drop-off from `login` to `view_ai`. If you had to choose ONE engineering intervention to improve conversion, what would it be and why?
5. (Stretch) Add a `cohort_retention` function that returns retention by week-of-signup.

---

**Read next:**
- `lessons/m18-leadership/L18-leadership-strategy.ipynb` -- how to present metrics to leadership
- `lessons/m16-fullstack/L16-fullstack-ai-app.ipynb` -- the AI app these metrics would measure
- `SYLLABUS.md` -- the 18 modules and 21-day plan